# Synthetic Cement Plant Digital Twin — Demonstration Environment

> **Synthetic Cement Plant Digital Twin — Demonstration Environment. All values are simulation
> estimates, not real factory measurements.**

This is the single Colab entry point of PRD v1.1.1 **Section 25**. It runs the whole pipeline in
the PRD's twelve-cell order — installation → configuration → simulation → dataset → validation →
training → evaluation → optimization → twin visualization → dashboard → the five **Section 28**
demo scenarios → export — by orchestrating the importable `src/` package unchanged (NFR-7). No
application logic lives in this notebook: every screen below is rendered by the project's own
renderers, every number comes out of the simulation, the models or the optimizer.

**To run:** `Runtime → Run all` (standard CPU runtime — NFR-1). On a fresh Colab runtime the
first cell clones the repository and nothing else needs to be installed by hand. Every demo cell
(section 11) is independently re-runnable once cells 1–8 have run.

**Honesty (PRD 21/30/31):** this dashboard reads a synthetic simulation. It is not connected to
any plant, it reads no plant instrument, and it writes no setpoint: every recommendation is
decision support for a human operator. No numeric confidence percentage appears anywhere;
recommendation quality is the categorical HIGH / MEDIUM / LOW only.

## 1 — Installation *(PRD §25 cell 1)*

Locate this repository (or clone it, when the notebook was opened in Colab rather than from a
checkout), make it importable, and `pip install` **only what is missing** beyond the runtime's
defaults. On a standard Colab runtime nothing is missing — every dependency of `src/` is
Colab-preinstalled (NFR-5) — so the install step is a check, not a download. The renderer layer
deliberately has zero extra dependencies: no plotly, no streamlit, no matplotlib.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Alisheikhalii/cement_digital_twin.git"


def _find_repo_root():
    """The nearest directory that *is* this repository (pyproject.toml + src/digital_twin)."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "digital_twin").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    return None


ROOT = _find_repo_root()
if ROOT is None:
    # Opened in Colab from GitHub: the notebook was loaded, the repository was not. Clone it
    # into the current working directory (relative - never a hard-coded absolute path).
    destination = Path.cwd() / "cement_digital_twin"
    if not destination.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(destination)], check=True
        )
    ROOT = destination

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# PRD 25 cell 1: install only what is missing. The import names differ from the pip names for
# scikit-learn and pyyaml, hence the explicit mapping rather than a guess.
REQUIRED = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
            "sklearn": "scikit-learn", "yaml": "pyyaml", "joblib": "joblib",
            "pyarrow": "pyarrow"}
missing = [pip_name for module, pip_name in REQUIRED.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

print(f"repository   {ROOT}")
print(f"python       {sys.version.split()[0]}")
print(f"installed    {missing or 'nothing - every dependency was already present'}")

## 2 — Configuration *(PRD §25 cell 2)*

Load and display every `configs/*.yaml` — the delay, balance, uncertainty and objective-weight
blocks included — and fix the run's seed and demo duration. The seed is read from
`configs/scenarios.yaml` and restated nowhere (NFR-4: identical config + seed ⇒ byte-identical
dataset). The duration override is PRD §25 cell 2's own "choose demo duration"; `None` keeps the
configured 30-day demo horizon, which visits all 14 PRD §11.4 regimes.

In [ ]:
from src.config import KILN, MILL, ML, OPTIMIZATION, SCENARIOS, load_config

configs = {name: load_config(name) for name in (KILN, MILL, ML, OPTIMIZATION, SCENARIOS)}
scenarios = configs[SCENARIOS]

SEED = int(scenarios.get_path("simulation.seed"))
DURATION_DAYS = None  # None = the configured demo duration (30 d); a number overrides it (PRD 25 cell 2)

configured_days = float(scenarios.get_path("simulation.duration_days"))
print(f"seed                 {SEED}   (from configs/scenarios.yaml)")
print(f"duration             {DURATION_DAYS if DURATION_DAYS is not None else configured_days} days"
      f"{' (configured)' if DURATION_DAYS is None else ' (override)'}")
print(f"prediction horizons  {list(configs[ML].get_path('prediction.horizons_min'))} min   (PRD 13.1)")
print(f"scheduled regimes    {len(scenarios.get_path('regime_schedule.regimes'))}   (PRD 11.4)")
print(f"configs loaded       {', '.join(sorted(configs))}")

## 3 — Simulation *(PRD §25 cell 3)*

`DatasetGenerator` owns exactly the loop PRD §25 cell 3 describes — the `ScenarioScheduler`
plans the run, `PlantTwin` is stepped once per minute with the energy/mass balance residuals
tracked at every step, and the `SensorModel` produces what a historian would have stored. This
cell runs that loop for the configured horizon and reads the conservation residuals back from
the run's own ground-truth frame. It states them as measured numbers; the tolerance verdicts
belong to the frozen conservation test suite, not to a notebook.

In [ ]:
from src.data_generation.generator import DatasetGenerator
from src.simulation.simulation_config import SimulationConfig

simulation = SimulationConfig.from_config(
    scenarios,
    **({} if DURATION_DAYS is None else {"duration_minutes": DURATION_DAYS * 1440.0}),
)
RUN = DatasetGenerator(simulation, scenarios=scenarios).run()

regimes = sorted(RUN.datasets["kiln"]["operating_regime"].unique())
print(f"exported rows        {len(RUN.index)}   ({simulation.duration_minutes / 1440.0:g} simulated days)")
print(f"regimes visited      {len(regimes)}: {', '.join(regimes)}")
print("balance residuals tracked per step (peak / mean |residual|):")
for dataset, frame in RUN.truth.items():
    for column in frame.columns:
        if column.endswith("_residual_pct"):
            values = frame[column].dropna().abs()
            print(f"  {dataset:5s} {column:30s} {values.max():9.4f} %   {values.mean():8.4f} %")

## 4 — Dataset generation *(PRD §25 cell 4)*

Export the run to `data/synthetic/` in the three PRD §11.6 forms: CSV, Parquet, and the JSON
config sidecar (which carries the seed and the schedule, and no wall-clock, so two runs of the
same seed produce byte-identical sidecars). The noise-free ground truth is exported *beside*
each dataset, never inside it.

In [ ]:
from src.data_generation.export import export_run

MANIFEST = export_run(RUN)
for dataset, written in MANIFEST.datasets.items():
    print(f"{dataset:5s} {', '.join(path.name for path in written)}")
    print(f"      truth: {', '.join(path.name for path in MANIFEST.truth[dataset])}")
print(f"sidecars {', '.join(path.name for path in MANIFEST.sidecars.values())} in {MANIFEST.directory}")

## 5 — Data validation *(PRD §25 cell 5)*

The FR-13 data-quality report (missing values, duplicates, constant sensors, spikes, drift, sync
issues) over both datasets, written to `reports/data_quality/`. The sensor model deliberately
injects dropout and drift, so the findings below are the report catching real, scheduled
imperfections — not a clean bill of health by construction.

In [ ]:
from src.data_processing.quality import report_run

QUALITY = report_run(RUN, write=True)
for name, report in QUALITY.items():
    counts = report.describe()["counts"]
    fired = {check: n for check, n in counts.items() if n}
    print(f"{name:5s} {report.rows} rows x {len(report.columns)} columns   severity: {report.severity}")
    print(f"      findings by check: {fired or 'none'}")

## 6 — ML training *(PRD §25 cell 6)*

Train Model A (one regressor per target × horizon, RandomForest + GradientBoosting selected on
held-out MAE, with the PRD 13.1.1 uncertainty methodology) and Model B (Isolation Forest + SPC),
then register them: `models/` joblib artifacts, the PRD 13.4 `registry.json`, and the PRD 22
metric reports. This is the full frozen training pipeline of `src.models.train` — the notebook
calls its public per-dataset functions (`train_model_a` / `train_model_b`, exactly what
`train_all` calls internally), it does not reimplement it.

*On a fresh clone this cell is what creates the model artifacts: the joblib blobs are not
version-controlled (regenerable from tracked source + configs), only `registry.json` is.
**Training is the pipeline's dominant cost — expect roughly half an hour** on a laptop-class
CPU (28 selected models, each with a bootstrap uncertainty ensemble, over 43 200 rows).*

*The cell is **resumable at the dataset level** (kiln, mill — two units; there are honestly no
sub-dataset percentages to report, only "1 of 2 datasets"): each dataset's results, artifacts
and registry entries are checkpointed by `src/notebook_support.py` after it completes. On
Google Colab, checkpoints persist to **Google Drive** — the mount authorization prompt appears
in this cell — so a Runtime disconnect or deletion loses at most the in-progress dataset;
completed datasets are reused, not retrained, and their metrics reports are rebuilt from the
checkpoint. Without Drive (declined mount, or running locally) checkpoints are stored
Runtime-/project-locally: they survive a restart in the same Runtime, but a Colab Runtime
**deletion** then loses them and both datasets retrain from scratch — that is the honest
limit of what is provided. A changed dataset or ML config invalidates a dataset's checkpoint
automatically (PRD 13.4 dataset hash + config digest).*


In [ ]:
from src.models.train import training_summary
from src.notebook_support import resumable_training

# Resumable per dataset (kiln, mill) - checkpoint/manifest logic lives in
# src/notebook_support.py (NFR-7: the cell only orchestrates).
TRAINING = resumable_training(RUN.datasets, truth=RUN.truth, simulation=RUN.provenance)
summary = training_summary(TRAINING)
print(f"datasets            {summary['datasets']}")
print(f"model A pairs       {summary['model_a_pairs']}   (one model per target x horizon)")
print(f"model A metric rows {summary['model_a_metric_rows']}")
print(f"model B splits      {summary['model_b_splits']}")


## 7 — Model evaluation *(PRD §25 cell 7)*

The per-target, per-horizon metrics PRD §22 requires, **both splits side by side** (AC-23): the
chronological split and the scenario holdout. Both were computed by cell 6 and written to
`reports/metrics/`; this cell reads them back and displays the selected models' rows. The
scenario-holdout columns are the harder, genuinely-unseen-regime test — read them as the honest
half of the table, not as a footnote.

In [ ]:
import json

import pandas as pd

from src import paths

model_a = json.loads((paths.REPORTS_METRICS_DIR / "model_a_horizon_metrics.json").read_text())
model_b = json.loads((paths.REPORTS_METRICS_DIR / "model_b_metrics.json").read_text())

rows = pd.DataFrame(model_a["rows"])
selected = rows[rows["selected"]].rename(columns={"horizon_min": "horizon [min]"})
display(
    selected.pivot_table(
        index=["dataset", "target", "horizon [min]"],
        columns=["split", "reference"],
        values=["mae", "rmse", "r2"],
        aggfunc="first",
    ).round(3)
)

print("model B methods:", list(model_b["methods"]))
print(f"model A report: {len(model_a['rows'])} rows, splits {sorted(set(rows['split']))} "
      f"(both PRD 13.3 splits present: AC-23)")

## 8 — Optimization *(PRD §25 cell 8)*

One example optimization, rendered as view J (AI Optimization): the PRD §14.4 `Recommendation`
with the hard-constraint gate results, the envelope/OOD validation, the multi-objective
breakdown, Recommendation Quality (HIGH/MEDIUM/LOW — never a percentage), the natural-language
reason, and the five PRD §14.5 baselines. If the optimizer refuses, the refusal is a first-class
display state with the blocking gates' own words.

This cell also builds the two objects every later cell shares: the **model layer** (loaded from
the artifacts cell 6 just registered — shared, not rebuilt per cell) and the dashboard **state**.

In [ ]:
from IPython.display import HTML

from app import build_document
from src.digital_twin.session import DashboardSession, build_model_layer
from src.digital_twin.state import VIEWS, DashboardState

LAYER = build_model_layer(RUN.datasets, scenarios=scenarios)
SESSION = DashboardSession.build(
    live=True, replay=False, scenarios=scenarios,
    models=LAYER, training=RUN.datasets,
)
STATE = DashboardState.from_session(SESSION)

WHAT_IF_IDS = frozenset({row[0] for row in VIEWS if row[1] == "what_if"} | {"what_if"})


class _WhatIfRequest:
    """The same duck-typed wrapper app.py's CLI uses: serve view I from a caller's mode and
    change set, delegate every other view. The engine and renderer do all the work."""

    def __init__(self, state, *, mode, deltas):
        self._state, self._mode, self._deltas = state, mode, dict(deltas)

    def view(self, view_id):
        if str(view_id) in WHAT_IF_IDS:
            return self._state.what_if(delta_fractions=self._deltas or None, mode=self._mode)
        return self._state.view(view_id)

    def capabilities(self):
        return self._state.capabilities()


def render(state, view_ids, **meta):
    """Render views through the project's own document assembler and display them inline."""
    html, timings = build_document(state, view_ids, settings=SESSION.settings, meta=meta or None)
    for view_id, seconds in timings.items():
        print(f"view {view_id:<4} {seconds:6.2f} s")
    for note in SESSION.notes():
        print(f"note: {note}")
    return html


def demo_session(regime=None, advance=0.0):
    """A demo session driven by one named regime (from configs/scenarios.yaml, PRD 28)."""
    session = DashboardSession.build(
        live=True, replay=False, scenarios=scenarios, regime=regime,
        models=LAYER, training=RUN.datasets,
    )
    if advance:
        session.provider.advance(advance)
    return DashboardState.from_session(session)


def save_demo(name, html):
    from src import paths

    path = paths.REPORTS_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(html, encoding="utf-8")
    print(f"wrote {path}")
    return path


HTML(render(STATE, ("J",), **{"PRD 25 cell": "8 - one example optimization (PRD 14.4)"}))

## 9 — Digital twin visualization *(PRD §25 cell 9)*

The animated Kiln and Mill twin views (B and E). These render the *live* simulation state — the
plant as the twin currently sees it, not the training dataset. The document assembler animates
them when given a live provider; here each is also written to `reports/notebook_twin_views.html`
so the notebook's own output survives being closed.

In [ ]:
html = render(STATE, ("B", "E"), **{"PRD 25 cell": "9 - digital twin visualization"})
save_demo("notebook_twin_views.html", html)
HTML(html)

## 10 — Interactive dashboard *(PRD §25 cell 10)*

The dashboard panels (A plant overview, C stability, F model quality, J optimization) followed by
the interactive what-if control panel. The widgets are thin orchestration only: the slider
ranges, steps, units and maximum deltas come from the What-if engine's own `slider()` spec
(PRD §16.3), the accept/reject verdict is the engine's EnvelopeReport, and view I is the
project's renderer. Moving a slider re-renders nothing else — the engine's answer is recomputed
on demand by the *Apply* button, not on every pixel.

In [ ]:
panels = render(STATE, ("A", "C", "F", "J"), **{"PRD 25 cell": "10 - dashboard panels"})
HTML(panels)

In [ ]:
import ipywidgets as widgets

baseline = STATE.what_if(mode="NORMAL")  # mode-independent slider spec (PRD 16.3)
SLIDERS = {}
for spec in baseline.sliders:
    SLIDERS[spec["name"]] = widgets.FloatSlider(
        description=f"{spec['name']} [{spec['unit']}]",
        value=spec["current"], min=spec["minimum"], max=spec["maximum"],
        step=spec["step"], continuous_update=False,
    )
MODE = widgets.ToggleButtons(options=["NORMAL", "EXPERIMENTAL"], description="Mode")
OUTPUT = widgets.Output()


def _slider_fraction(name, spec):
    """Slider value -> the signed fraction of current value the engine expects (the same
    percent-of-current semantics app.py's --change flag uses)."""
    current = spec["current"]
    return 0.0 if current == 0 else (SLIDERS[name].value - current) / current


def _sync_bounds(*_):
    """Re-clamp every slider to the active mode's bounds (Experimental widens them, PRD 16.4)."""
    spec_by_name = {spec["name"]: spec for spec in STATE.what_if(mode=MODE.value).sliders}
    for name, slider in SLIDERS.items():
        spec = spec_by_name[name]
        slider.min, slider.max = spec["minimum"], spec["maximum"]
        slider.step = spec["step"]
        slider.value = min(max(slider.value, spec["minimum"]), spec["maximum"])


def _apply(_):
    spec_by_name = {spec["name"]: spec for spec in STATE.what_if(mode=MODE.value).sliders}
    deltas = {name: _slider_fraction(name, spec_by_name[name]) for name in SLIDERS}
    deltas = {name: value for name, value in deltas.items() if value} or None
    OUTPUT.clear_output(wait=True)
    with OUTPUT:
        request = _WhatIfRequest(STATE, mode=MODE.value, deltas=deltas or {})
        html = render(request, ("I",), **{"PRD 25 cell": "10 - interactive what-if",
                                          "what-if mode": MODE.value})
        display(HTML(html))
        display(HTML(html))


MODE.observe(_sync_bounds, names="value")
APPLY = widgets.Button(description="Apply what-if", button_style="primary")
APPLY.on_click(_apply)
display(widgets.VBox([MODE, *SLIDERS.values(), APPLY, OUTPUT]))

## 11 — Demo scenarios *(PRD §28, §25 cell 11)*

The five PRD §28 demos. Each is one self-contained, re-runnable cell — no manual setup beyond
cells 1–8 having run. Every scenario comes from `configs/scenarios.yaml` (never an invented
number), every screen is a project renderer, and every verdict is the engine's.

**Honest scope note (Demo 3):** PRD §28 describes Demo 3 as "inject a low-oxygen condition".
The repository has no *injection* mechanism (FR-10 is unimplemented); what exists is the
scheduler. Demo 3 therefore drives the configured low-oxygen *regime* via
`--scenario`-equivalent scheduling and the notebook states this substitution openly rather than
fabricating an inject API.

In [ ]:
# Demo 1 - Normal Operation (PRD 28.1): the startup block settles into normal medium production,
# views B + A. Regime names come from configs/scenarios.yaml - never restated here.
state = demo_session(regime="Normal - medium production", advance=30.0)
html = render(state, ("B", "A"), **{"PRD 28 demo": "1 - normal operation",
                                    "regime": "Normal - medium production"})
save_demo("notebook_demo_1_normal_operation.html", html)
HTML(html)

In [ ]:
# Demo 2 - Energy Optimization (PRD 28.2): the high fuel condition, view J, with the model's
# recommendation and its gates. The optimizer decides; this cell only displays the decision.
state = demo_session(regime="High fuel condition", advance=30.0)
html = render(state, ("J",), **{"PRD 28 demo": "2 - energy optimization",
                                "regime": "High fuel condition"})
save_demo("notebook_demo_2_energy_optimization.html", html)
HTML(html)

In [ ]:
# Demo 3 - Low Oxygen (PRD 28.3): the configured low-oxygen regime, views B + H.
# PRD 28 wording says "inject"; the codebase has no inject mechanism (FR-10 backend gap), so
# this drives the *scheduled* low-oxygen condition instead - same physics, honestly labelled.
state = demo_session(regime="Low oxygen condition", advance=30.0)
html = render(state, ("B", "H"), **{"PRD 28 demo": "3 - low oxygen condition",
                                    "regime": "Low oxygen condition",
                                    "mechanism": "scheduled regime (no inject API exists - FR-10 gap)"})
save_demo("notebook_demo_3_low_oxygen.html", html)
HTML(html)

In [ ]:
# Demo 4 - Mill Optimization (PRD 28.4): separator speed +5% of its current value in NORMAL
# mode, views E + I. The engine owns the verdict; view I shows accept or reject either way.
request = _WhatIfRequest(STATE, mode="NORMAL", deltas={"separator_speed_rpm": 0.05})
html = render(request, ("E", "I"), **{"PRD 28 demo": "4 - mill optimization",
                                      "change": "separator_speed_rpm +5% (NORMAL)"})
save_demo("notebook_demo_4_mill_optimization.html", html)
HTML(html)

In [ ]:
# Demo 5 - What-if Analysis (PRD 28.5): the same -5% kiln fuel change under both modes.
# NORMAL: inside the envelope -> PASS. EXPERIMENTAL: -25% fuel is outside the operating
# envelope -> the engine refuses, and view I shows the refusal as a first-class answer.
from IPython.display import display as _display

for mode, delta in (("NORMAL", -0.05), ("EXPERIMENTAL", -0.25)):
    request = _WhatIfRequest(STATE, mode=mode, deltas={"kiln_fuel_rate_tph": delta})
    html = render(request, ("I",), **{"PRD 28 demo": "5 - what-if analysis",
                                      "change": f"kiln_fuel_rate_tph {delta:.0%}",
                                      "mode": mode})
    _display(HTML(html))
    save_demo(f"notebook_demo_5_what_if_{mode.lower()}.html", html)

## 12 — Export results *(PRD §25 cell 12)*

Bundle everything the notebook produced into `reports/notebook_export/`: the metric and
data-quality reports, the twin and demo HTML views, and the dataset sidecars. On Colab, use the
Files browser (folder icon, left edge) to download the bundle — or `files.download()` after
uncommenting the last line.

In [ ]:
import zipfile

from src import paths

EXPORT_DIR = paths.REPORTS_DIR / "notebook_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE = EXPORT_DIR / "cement_digital_twin_notebook_export.zip"

members = [
    *sorted(paths.REPORTS_METRICS_DIR.glob("*.json")),
    *sorted(paths.REPORTS_DATA_QUALITY_DIR.glob("*.json")),
    *sorted(paths.REPORTS_DIR.glob("notebook_*.html")),
    *sorted(MANIFEST.sidecars.values()),
]
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in members:
        archive.write(path, path.relative_to(ROOT))
print(f"{BUNDLE}")
for path in members:
    print(f"  {path.relative_to(ROOT)}")

# from google.colab import files; files.download(str(BUNDLE))  # uncomment on Colab